#####**Bloco 1 - Instalação das bibliotecas para conectar ao Google**

In [ ]:
# --- INSTALAÇÃO LIMPA ---
# !pip install --upgrade gspread google-auth pandas pandas-gbq

# print("Bibliotecas atualizadas!")

#####**Bloco 2 - Autenticação e Conexão** (Onde Apresentamos o "Crachá" do Robô)

In [ ]:
import pandas as pd
import gspread
from google.oauth2 import service_account
from google.colab import userdata
from datetime import datetime
import json

# CONFIGURAÇÕES
NOME_DA_PLANILHA = 'Planilha LavourJoias 2.0'
NOME_DA_ABA = 'Página1'
ID_PROJETO_CLOUD = 'shopee-analytics-projeto'
NOME_DATASET_BQ = 'loja_joias'
NOME_TABELA_BQ = 'historico_estoque'

print("Acessando cofre de credenciais e renovando permissões...")

try:
    # Buscando a chave
    json_info = userdata.get('GCP_JSON_KEY')
    info_dict = json.loads(json_info)

    # CRIANDO AS CREDENCIAIS COM O ESCOPO DO DRIVE
    credentials = service_account.Credentials.from_service_account_info(
        info_dict,
        scopes=[
            "https://www.googleapis.com/auth/spreadsheets",
            "https://www.googleapis.com/auth/bigquery",
            "https://www.googleapis.com/auth/drive"
        ]
    )

    # Autenticar no Google Sheets
    gc = gspread.authorize(credentials)
    print("✅ Autenticação renovada com permissão de Drive!")

except Exception as e:
    print(f"Erro na autenticação: {e}")

Acessando cofre de credenciais e renovando permissões...
✅ Autenticação renovada com permissão de Drive!


#####**Bloco 3 - Extração / Extract** ( O Robô Abre a Planilha e Puxa os Dados Brutos )

In [ ]:
# --- EXTRAÇÃO ---
NOME_DA_PLANILHA = 'Planilha Lavourjoias 2.0'
NOME_DA_ABA = 'Estoque'

print(f"1. Acessando a planilha '{NOME_DA_PLANILHA}'...")

try:
    sh = gc.open(NOME_DA_PLANILHA)
    worksheet = sh.worksheet(NOME_DA_ABA)

    # Pega os dados brutos para evitar erros de cabeçalho
    rows = worksheet.get_all_values()

    # Cria a tabela usando a primeira linha como cabeçalho
    df = pd.DataFrame(rows[1:], columns=rows[0])

    # Remove as colunas vazias
    df = df.loc[:, df.columns != '']

    print(f"   > Sucesso! {len(df)} linhas carregadas.")
    display(df.head(3))

except Exception as e:
    print(f"ERRO: {e}")

1. Acessando a planilha 'Planilha Lavourjoias 2.0'...
   > Sucesso! 35 linhas carregadas.


,ID do item,Nome do item,Tipo,Preço,Entrada,Saída,Estoque,Status
0,,Colar Veneziana 40cm,Prata,"R$ 17,00",7,0,7,Em estoque
1,,Colar Veneziana 45cm,Prata,"R$ 20,00",8,3,5,Em estoque
2,,Colar Veneziana 50cm,Prata,"R$ 25,00",6,0,6,Em estoque


#####**Bloco 4 - Transformação / Transform** (Copiar e Não Perder os Dados dos Dias Anteriores)

In [ ]:
# --- BLOCO 4: TRANSFORMAÇÃO (COM OTIMIZAÇÃO DE COLUNAS) ---
from datetime import datetime
import pandas as pd

print("2. Tratando e otimizando os dados...")

# 1. Carimbo de data
df['data_carga'] = datetime.now()

# 2. Função de limpeza de moeda
def limpar_moeda(valor):
    if isinstance(valor, str):
        limpo = valor.replace('R$', '').replace('.', '').replace(',', '.').strip()
        return float(limpo) if limpo else 0.0
    return valor

# 3. Limpeza da coluna Preço
if 'Preço' in df.columns:
    df['Preço'] = df['Preço'].apply(limpar_moeda)

# 4. Conversão de números
colunas_numericas = ['Entrada', 'Saída', 'Estoque']
for col in colunas_numericas:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

# ===============================================================
# 5. OTIMIZAÇÃO: REMOVER COLUNAS QUE NÃO QUERO NO BANCO DE DADOS
# ===============================================================
colunas_para_excluir = ['Status']

df = df.drop(columns=colunas_para_excluir, errors='ignore')

print(f"   > Coluna(s) {colunas_para_excluir} removida(s) com sucesso para economizar espaço!")

print("-" * 30)
display(df.head(2))

2. Tratando e otimizando os dados...
   > Coluna(s) ['Status'] removida(s) com sucesso para economizar espaço!
------------------------------


,ID do item,Nome do item,Tipo,Preço,Entrada,Saída,Estoque,data_carga
0,,Colar Veneziana 40cm,Prata,17.0,7,0,7,2026-03-09 23:25:26.267161
1,,Colar Veneziana 45cm,Prata,20.0,8,3,5,2026-03-09 23:25:26.267161


#####**Bloco 5 - Carga / Load + Criação Automática** (Verifica se Dataset existe no BigQuery, salvando as informações. Se não, ele cria)

In [ ]:
# --- CARGA ---
from google.cloud import bigquery

ID_PROJETO_CLOUD = 'data-shopee-estudos'

NOME_DATASET_BQ = 'loja_joias'
NOME_TABELA_BQ = 'historico_estoque'

print(f"3. Conectando ao BigQuery no projeto '{ID_PROJETO_CLOUD}'...")

try:
    # Cliente do BigQuery
    client_bq = bigquery.Client(credentials=credentials, project=ID_PROJETO_CLOUD)

    # Verifica/Cria o Dataset
    dataset_id = f"{ID_PROJETO_CLOUD}.{NOME_DATASET_BQ}"
    try:
        client_bq.get_dataset(dataset_id)
        print(f"   > Dataset '{NOME_DATASET_BQ}' encontrado.")
    except:
        print(f"   > Dataset '{NOME_DATASET_BQ}' não existe. Criando...")
        dataset = bigquery.Dataset(dataset_id)
        dataset.location = "southamerica-east1"
        client_bq.create_dataset(dataset)
        print("   > Dataset criado!")

    # Envia os dados
    print(f"   > Salvando {len(df)} linhas na tabela '{NOME_TABELA_BQ}'...")

    df.to_gbq(
        destination_table=f'{NOME_DATASET_BQ}.{NOME_TABELA_BQ}',
        project_id=ID_PROJETO_CLOUD,
        credentials=credentials,
        if_exists='append'
    )

    print("✅ SUCESSO ABSOLUTO! Seus dados estão na nuvem.")

except Exception as e:
    print(f"❌ ERRO: {e}")
    print("Dica: Verifique se o ID_PROJETO_CLOUD está igual ao 'project_id' do seu arquivo JSON.")

3. Conectando ao BigQuery no projeto 'data-shopee-estudos'...
   > Dataset 'loja_joias' encontrado.
   > Salvando 35 linhas na tabela 'historico_estoque'...


/tmp/ipykernel_1144/2958689935.py:30: FutureWarning: to_gbq is deprecated and will be removed in a future version. Please use pandas_gbq.to_gbq instead: https://pandas-gbq.readthedocs.io/en/latest/api.html#pandas_gbq.to_gbq
  df.to_gbq(
100%|██████████| 1/1 [00:00<00:00, 9218.25it/s]

✅ SUCESSO ABSOLUTO! Seus dados estão na nuvem.
